In [14]:
#!/usr/bin/env python
"""
Re‐implementation of key Caver 3.0 functions in Python using a power–diagram 
approach for clearance calculation and hierarchical tunnel clustering based on 
the average throughput penalty function. Docking is performed with pycaverdock.

The pipeline:
  • Ingests protein (and optionally trajectory) and ligand files.
  • Computes protein effective radii (van der Waals + probe radius).
  • Generates a 3D grid over the protein.
  • For each grid point, computes clearance using a weighted (power diagram) metric.
  • Builds a connectivity graph (via grid neighbors) and propagates from a chosen 
    start coordinate to find candidate tunnels.
  • The start coordinate can be determined by:
        - "atom": using an active‐site atom (then the closest accessible point is used).
        - "ligand_frame0": using the ligand’s center–of–mass from frame 0.
        - "clustering": using DBSCAN clustering around an active–site atom.
  • For each candidate tunnel, computes an average clearance (“throughput”) and length.
  • Clusters the tunnels hierarchically (within a frame and over a trajectory) using a 
    penalty function combining spatial overlap and throughput difference.
  • Uses pycaverdock for docking along the tunnel.
  • Exports the tunnel data into a pandas DataFrame for downstream analysis.
  
Requirements:
  - MDAnalysis, SciPy, NumPy, matplotlib, py3Dmol, scikit-learn, pandas, pycaverdock,
    and OpenBabel (pybel and openbabel).
"""

import os, tempfile, heapq
import numpy as np
import MDAnalysis as mda
from MDAnalysis.lib import distances
from scipy.spatial import KDTree
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
import py3Dmol
from openbabel import pybel
from openbabel import openbabel as ob
from multiprocessing import Pool, cpu_count
import pandas as pd
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform
import pycaverdock  # Docking package

### Protein and Ligand Ingestion Functions ###
def convert_to_pdbqt_ligand(input_file, output_file=None):
    ext = os.path.splitext(input_file)[1].lower()[1:]
    if output_file is None:
        base = os.path.splitext(input_file)[0]
        output_file = base + ".pdbqt"
    mols = list(pybel.readfile(ext, input_file))
    if not mols:
        raise ValueError(f"No molecules found in {input_file}")
    mol = mols[0]
    mol.addh()       # Add hydrogens (for pH ~7.4)
    mol.make3D()     # Generate 3D geometry
    mol.calccharges(model='gasteiger')
    mol.write("pdbqt", output_file, overwrite=True)
    return output_file

def convert_to_pdbqt_protein(input_file, output_file=None):
    if os.path.splitext(input_file)[1].lower() != ".pdb":
        raise ValueError("Protein input must be a PDB file.")
    if output_file is None:
        base = os.path.splitext(input_file)[0]
        output_file = base + "_rigid.pdbqt"
    conv = ob.OBConversion()
    conv.SetInFormat("pdb")
    conv.SetOutFormat("pdbqt")
    conv.AddOption("r", ob.OBConversion.OUTOPTIONS)  # rigid output
    obmol = ob.OBMol()
    conv.ReadFile(obmol, input_file)
    obmol.CorrectForPH()  # optional
    obmol.AddHydrogens()  # ensure hydrogens are present
    conv.WriteFile(obmol, output_file)
    return output_file

def universe_frame_to_pdbqt(u, output_file):
    temp_pdb = tempfile.NamedTemporaryFile(delete=False, suffix=".pdb").name
    with mda.Writer(temp_pdb) as W:
        W.write(u.atoms)
    pdbqt_file = convert_to_pdbqt_protein(temp_pdb, output_file)
    os.remove(temp_pdb)
    return pdbqt_file

def ingest_files(protein_input, ligand_input):
    protein_ext = os.path.splitext(protein_input)[1].lower()
    ligand_ext = os.path.splitext(ligand_input)[1].lower()
    if protein_ext != ".pdbqt":
        protein_pdbqt = convert_to_pdbqt_protein(protein_input)
    else:
        protein_pdbqt = protein_input
    if ligand_ext != ".pdbqt":
        ligand_pdbqt = convert_to_pdbqt_ligand(ligand_input)
    else:
        ligand_pdbqt = ligand_input
    return protein_pdbqt, ligand_pdbqt

### Pocket Identification and Start Coordinate Determination ###
def get_protein_only(u):
    return u.select_atoms("protein")

def pocket_atom(active_atom):
    """Return the coordinate of the active-site atom."""
    return active_atom.position

def pocket_ligand_frame0(u, ligand_sel):
    """Determine the start coordinate as the ligand’s center–of–mass in frame 0."""
    ligand = u.select_atoms(ligand_sel)
    if len(ligand) == 0:
        raise ValueError("No ligand found using the provided selection in frame 0.")
    return ligand.center_of_mass()

def pocket_clustering(u, active_atom, search_radius=12.0, eps=3.0, min_samples=5):
    """
    Determine the pocket center by clustering protein atoms near the active site.
    Returns a centroid and the pocket atoms.
    """
    protein = get_protein_only(u)
    nearby = protein.select_atoms("around %f index %d" % (search_radius, active_atom.index))
    if active_atom not in nearby:
        active_ag = u.select_atoms("index %d" % active_atom.index)
        nearby = nearby.union(active_ag)
    coords = nearby.positions
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(coords)
    labels = db.labels_
    unique_labels = set(labels)
    if -1 in unique_labels:
        unique_labels.remove(-1)
    if not unique_labels:
        raise ValueError("No clusters found; try adjusting eps/min_samples.")
    centroids = {label: coords[labels == label].mean(axis=0) for label in unique_labels}
    active_pos = active_atom.position
    best_label = min(centroids.keys(), key=lambda l: np.linalg.norm(centroids[l] - active_pos))
    pocket_atoms = nearby[labels == best_label]
    return centroids[best_label], pocket_atoms

def get_pocket_center(u, method, active_site_atom_sel, ligand_sel, **kwargs):
    """
    Determine the pocket/tunnel start coordinate.
      - "atom": use the active site atom’s position (later adjusted to the closest accessible grid point).
      - "ligand_frame0": use the ligand’s center-of-mass from frame 0.
      - "clustering": use DBSCAN clustering on protein atoms around the active site.
    """
    if method == "atom":
        active_atoms = u.select_atoms(active_site_atom_sel)
        if len(active_atoms) == 0:
            raise ValueError("No active site atom found!")
        return pocket_atom(active_atoms[0]), None
    elif method == "ligand_frame0":
        u.trajectory[0]
        return pocket_ligand_frame0(u, ligand_sel), None
    elif method == "clustering":
        active_atoms = u.select_atoms(active_site_atom_sel)
        if len(active_atoms) == 0:
            raise ValueError("No active site atom found!")
        return pocket_clustering(u, active_atoms[0], **kwargs)
    else:
        raise ValueError("Unknown pocket method: choose from 'atom', 'ligand_frame0', or 'clustering'.")

### Grid and Power–Diagram Based Clearance Functions ###
def compute_grid_bounds(u, margin):
    protein = get_protein_only(u)
    coords = protein.positions
    return coords.min(axis=0) - margin, coords.max(axis=0) + margin

def generate_grid(min_coords, max_coords, spacing):
    xs = np.arange(min_coords[0], max_coords[0] + spacing, spacing)
    ys = np.arange(min_coords[1], max_coords[1] + spacing, spacing)
    zs = np.arange(min_coords[2], max_coords[2] + spacing, spacing)
    grid = np.meshgrid(xs, ys, zs, indexing='ij')
    grid_points = np.vstack([g.ravel() for g in grid]).T
    grid_shape = (len(xs), len(ys), len(zs))
    return grid_points, grid_shape

# Default van der Waals radii (in Å)
VDW_RADII = {'C': 1.7, 'N': 1.55, 'O': 1.52, 'S': 1.8, 'H': 1.2, 'P': 1.8}

def get_effective_radii(protein, probe_radius):
    radii = []
    for atom in protein:
        try:
            elem = atom.element.strip()
        except AttributeError:
            elem = atom.name[0]
        r_vdw = VDW_RADII.get(elem, 1.7)
        r_eff = r_vdw + probe_radius
        radii.append(r_eff)
    return np.array(radii)

def compute_clearance_powerdiagram(u, grid_points, probe_radius):
    """
    Compute clearance for each grid point using a power–diagram metric:
      clearance = max( min_p(||x - p|| - (r_vdw + probe_radius)), 0 )
    """
    protein = get_protein_only(u)
    protein_coords = protein.positions
    effective_radii = get_effective_radii(protein, probe_radius)
    tree = KDTree(protein_coords)
    clearances = np.empty(len(grid_points))
    # Query a few nearest atoms for each grid point
    for i, pt in enumerate(grid_points):
        dists, idxs = tree.query(pt, k=5)
        if np.isscalar(dists):
            dists = np.array([dists])
            idxs = np.array([idxs])
        cand = np.inf
        for d, idx in zip(dists, idxs):
            r_eff = effective_radii[idx]
            val = d - r_eff
            if val < cand:
                cand = val
        clearances[i] = max(cand, 0)
    return clearances

def build_3d_arrays(grid_points, clearances, grid_shape):
    grid_3d = grid_points.reshape(grid_shape + (3,))
    clearance_3d = clearances.reshape(grid_shape)
    return grid_3d, clearance_3d

def is_boundary(idx, grid_shape):
    i, j, k = idx
    nx, ny, nz = grid_shape
    return (i == 0 or i == nx - 1 or j == 0 or j == ny - 1 or k == 0 or k == nz - 1)

### Tunnel Finding using a Power–Diagram Approach ###
def find_tunnels(u, grid_3d, clearance_3d, accessible_3d, start_coord):
    """
    Propagate from the start coordinate using a bottleneck (max–min clearance) approach.
    When using an "atom" start, the given start_coord (active site) is adjusted to the closest
    accessible grid point.
    
    Returns a list of candidate tunnels. Each tunnel is a dict containing:
      - 'path': Nx3 array of coordinates (the tunnel path)
      - 'clearance_profile': clearance along the path
      - 'avg_clearance': average clearance (throughput)
      - 'length': tunnel length
    """
    grid_shape = clearance_3d.shape
    flat_grid = grid_3d.reshape(-1, 3)
    # Determine the starting grid index closest to the provided start_coord
    start_flat_idx = np.argmin(np.linalg.norm(flat_grid - start_coord, axis=1))
    start_idx = np.unravel_index(start_flat_idx, grid_shape)
    # If the closest grid point is not accessible, find the nearest accessible point.
    if not accessible_3d[start_idx]:
        accessible_indices = np.argwhere(accessible_3d)
        if accessible_indices.size == 0:
            raise ValueError("No accessible grid point found!")
        accessible_coords = np.array([grid_3d[tuple(idx)] for idx in accessible_indices])
        distances = np.linalg.norm(accessible_coords - start_coord, axis=1)
        best_idx = accessible_indices[np.argmin(distances)]
        start_idx = tuple(best_idx)
    
    # Propagation using a modified Dijkstra approach (maximizing the minimum clearance along the path)
    bottleneck = -np.ones(grid_shape) * np.inf
    bottleneck[start_idx] = clearance_3d[start_idx]
    parent = {start_idx: None}
    queue = [(-bottleneck[start_idx], start_idx)]
    neighbor_offsets = [(-1, 0, 0), (1, 0, 0),
                        (0, -1, 0), (0, 1, 0),
                        (0, 0, -1), (0, 0, 1)]
    while queue:
        curr_val_neg, curr_idx = heapq.heappop(queue)
        curr_val = -curr_val_neg
        i, j, k = curr_idx
        for di, dj, dk in neighbor_offsets:
            ni, nj, nk = i + di, j + dj, k + dk
            if 0 <= ni < grid_shape[0] and 0 <= nj < grid_shape[1] and 0 <= nk < grid_shape[2]:
                neighbor_idx = (ni, nj, nk)
                if not accessible_3d[neighbor_idx]:
                    continue
                new_val = min(curr_val, clearance_3d[neighbor_idx])
                if new_val > bottleneck[neighbor_idx]:
                    bottleneck[neighbor_idx] = new_val
                    parent[neighbor_idx] = curr_idx
                    heapq.heappush(queue, (-new_val, neighbor_idx))
    
    # Trace back paths from each accessible boundary point.
    tunnels = []
    boundary_list = [idx for idx in np.ndindex(grid_shape) if is_boundary(idx, grid_shape) and accessible_3d[idx]]
    for b_idx in boundary_list:
        if b_idx in parent:
            path_idx = []
            curr = b_idx
            while curr is not None:
                path_idx.append(curr)
                curr = parent.get(curr)
            path_idx.reverse()
            tunnel_path = np.array([grid_3d[idx] for idx in path_idx])
            clearance_profile = np.array([clearance_3d[idx] for idx in path_idx])
            avg_clearance = np.mean(clearance_profile)
            length = np.sum(np.linalg.norm(np.diff(tunnel_path, axis=0), axis=1))
            tunnels.append({'path': tunnel_path,
                            'clearance_profile': clearance_profile,
                            'avg_clearance': avg_clearance,
                            'length': length})
    return tunnels

### Tunnel Clustering (Hierarchical based on Average Throughput Penalty) ###
def tunnel_similarity(tunnel1, tunnel2, threshold=1.0):
    """
    Computes a custom distance between two tunnels.
    Combines:
      - Overlap: fraction of points in one tunnel near any point in the other.
      - Throughput difference: absolute difference in average clearance.
    Lower value indicates more similarity.
    """
    path1 = tunnel1['path']
    path2 = tunnel2['path']
    thresh = threshold  # spatial threshold in Å
    
    count1 = np.sum([np.any(np.linalg.norm(path2 - p, axis=1) < thresh) for p in path1])
    overlap1 = count1 / len(path1)
    
    count2 = np.sum([np.any(np.linalg.norm(path1 - p, axis=1) < thresh) for p in path2])
    overlap2 = count2 / len(path2)
    
    overlap_fraction = (overlap1 + overlap2) / 2.0
    throughput_diff = abs(tunnel1['avg_clearance'] - tunnel2['avg_clearance'])
    return (1 - overlap_fraction) + throughput_diff

def cluster_tunnels(tunnels, distance_threshold=0.5):
    """
    Hierarchically cluster a list of tunnels using the custom similarity metric.
    """
    n = len(tunnels)
    if n == 0:
        return []
    dist_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            d = tunnel_similarity(tunnels[i], tunnels[j])
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d
    condensed = squareform(dist_matrix)
    Z = sch.linkage(condensed, method='average')
    cluster_labels = sch.fcluster(Z, t=distance_threshold, criterion='distance')
    for i, tunnel in enumerate(tunnels):
        tunnel['cluster'] = cluster_labels[i]
    return tunnels

### Processing a Single Frame or Entire Trajectory ###
def process_frame(u, grid_spacing, probe_radius, grid_margin, active_site_atom_sel,
                  start_method, ligand_sel, cluster_kwargs, distance_threshold=0.5):
    """
    Process one frame: build grid, compute clearance, find tunnels,
    and cluster them. Returns a list of tunnels with frame info.
    """
    protein = get_protein_only(u)
    active_atoms = u.select_atoms(active_site_atom_sel)
    if len(active_atoms) == 0:
        raise ValueError("No active site atom found!")
    
    # Determine the start coordinate based on the selected method.
    start_coord, pocket_atoms = get_pocket_center(u, start_method, active_site_atom_sel, ligand_sel, **cluster_kwargs)
    print(start_coord)


    min_coords, max_coords = compute_grid_bounds(u, grid_margin)
    grid_points, grid_shape = generate_grid(min_coords, max_coords, grid_spacing)
    clearances = compute_clearance_powerdiagram(u, grid_points, probe_radius)
    grid_3d, clearance_3d = build_3d_arrays(grid_points, clearances, grid_shape)
    accessible_3d = (clearance_3d > 0)
    
    tunnels = find_tunnels(u, grid_3d, clearance_3d, accessible_3d, start_coord)
    tunnels = cluster_tunnels(tunnels, distance_threshold)
    for tunnel in tunnels:
        tunnel['frame'] = u.trajectory.frame
    return tunnels

def process_trajectory(u, grid_spacing, probe_radius, grid_margin, active_site_atom_sel,
                       start_method, ligand_sel, cluster_kwargs, distance_threshold=0.5):
    """
    Process every frame in the trajectory to obtain tunnels.
    Returns a list of tunnel dictionaries.
    """
    all_tunnels = []
    for ts in u.trajectory:
        try:
            frame_tunnels = process_frame(u, grid_spacing, probe_radius, grid_margin,
                                          active_site_atom_sel, start_method, ligand_sel,
                                          cluster_kwargs, distance_threshold)
            all_tunnels.extend(frame_tunnels)
        except Exception as e:
            print(f"Frame {u.trajectory.frame}: {e}")
    return all_tunnels

def tunnels_to_dataframe(tunnels):
    """
    Convert a list of tunnel dictionaries into a pandas DataFrame.
    """
    rows = []
    for i, tunnel in enumerate(tunnels):
        row = {'tunnel_id': i,
               'frame': tunnel.get('frame', None),
               'cluster': tunnel.get('cluster', None),
               'avg_clearance': tunnel.get('avg_clearance', None),
               'length': tunnel.get('length', None),
               'path': tunnel.get('path', None),
               'clearance_profile': tunnel.get('clearance_profile', None)}
        rows.append(row)
    return pd.DataFrame(rows)

### Visualization Functions ###
def visualize_tunnel_py3dmol(tunnel_path, protein_vis_file, pocket_atoms=None, tunnel_radii=None):
    with open(protein_vis_file, 'r') as f:
        pdb_data = f.read()
    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_data, 'pdb')
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    for i, point in enumerate(tunnel_path):
        rad = float(tunnel_radii[i]) if tunnel_radii is not None else 0.5
        view.addSphere({
            'center': {'x': float(point[0]), 'y': float(point[1]), 'z': float(point[2])},
            'radius': rad, 'color': 'red', 'opacity': 0.8
        })
    if pocket_atoms is not None and len(pocket_atoms) > 0:
        for atom in pocket_atoms:
            view.addSphere({
                'center': {'x': float(atom.position[0]),
                           'y': float(atom.position[1]),
                           'z': float(atom.position[2])},
                'radius': 0.5, 'color': 'blue', 'opacity': 0.3
            })
    view.zoomTo()
    view.show()


### Main Workflow ###
def main():
    protein_input = "1be0.pdb"          # Protein input file (for visualization)
    ligand_input = "EtOH.sdf"           # Ligand input file (SDF/MOL2)
    traj_file = None                  # Set to a trajectory file if available
    
    # Start coordinate determination parameters:
    active_site_atom_sel = "resid 124 and name OD2"
    start_method = "clustering"  # Choose from 'atom', 'ligand_frame0', 'clustering'
    ligand_sel = "resname LIG"  # used if start_method == "ligand_frame0"
    cluster_kwargs = {'search_radius': 12.0, 'eps': 3.0, 'min_samples': 5}
    
    # Grid and clearance parameters:
    grid_spacing = 0.6
    probe_radius = 0.9
    grid_margin = 2
    distance_threshold = 0.5  # for tunnel clustering
    
    # Ingest docking files (PDBQT conversion)
    protein_dock, ligand_dock = ingest_files(protein_input, ligand_input)
    
    # Load protein (and trajectory if available)
    u = mda.Universe(protein_input, traj_file) if traj_file else mda.Universe(protein_input)
    
    # Process either a single frame or a trajectory:
    if traj_file is None:
        # Process a single frame:
        tunnels = process_frame(u, grid_spacing, probe_radius, grid_margin,
                                active_site_atom_sel, start_method, ligand_sel, cluster_kwargs,
                                distance_threshold)
        if tunnels:
            best_tunnel = max(tunnels, key=lambda t: t['avg_clearance'])
            radii = best_tunnel['clearance_profile']
            dists_profile = np.linspace(0, np.linalg.norm(best_tunnel['path'][-1] - best_tunnel['path'][0]),
                                        len(radii))
            plt.figure(figsize=(8, 4))
            plt.plot(dists_profile, 2 * radii, marker='o')
            plt.xlabel("Distance along tunnel (Å)")
            plt.ylabel("Estimated tunnel diameter (Å)")
            plt.title("Tunnel Geometry Profile")
            plt.grid(True)
            plt.show()
            visualize_tunnel_py3dmol(best_tunnel['path'], protein_input, pocket_atoms=None, tunnel_radii=radii)
            
            df = tunnels_to_dataframe(tunnels)
            print(df)
        else:
            print("No tunnels found in the static structure.")
    
    else:
        # Process the entire trajectory:
        tunnels = process_trajectory(u, grid_spacing, probe_radius, grid_margin,
                                     active_site_atom_sel, start_method, ligand_sel, cluster_kwargs,
                                     distance_threshold)
        df = tunnels_to_dataframe(tunnels)
        print(df)
        # Further processing of df with pandas can be done here.

if __name__ == "__main__":
    main()


*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is 1be0.pdb)



[29.353716 27.286955 31.017185]
No tunnels found in the static structure.
